In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
!unzip /content/sample_data/titanic.zip

Archive:  /content/sample_data/titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [11]:
test_data= pd.read_csv('/content/test.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
train_data = pd.read_csv('/content/train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rateF = sum(women) / len(women)

print("% is", rateF*100)

% is 74.20382165605095


In [14]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rateM = sum(men) / len(men)

print("% is", rateM*100)

% is 18.890814558058924


In [15]:
for df in [train_data, test_data]:
    df['Title'] = df.Name.str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

title_medians = df.groupby('Title')['Age'].median()

for df in [train_data, test_data]:
    for title in title_medians.index:
        df.loc[(df['Age'].isnull()) & (df['Title'] == title), 'Age'] = title_medians[title]

In [16]:
train_median_age = train_data["Age"].median()
train_data["Age"] = train_data["Age"].fillna(train_median_age)
test_data["Age"] = test_data["Age"].fillna(train_median_age)

bins = [0, 12, 18, 30, 60, 100]
labels = ["Child", "Teenager", "Young Adult", "Adult", "Senior"]

train_data["AgeGroup"] = pd.cut(train_data["Age"], bins = bins, labels = labels)
test_data["AgeGroup"] = pd.cut(test_data["Age"], bins = bins, labels = labels)

In [17]:
train_data["FamilySize"] = train_data["Parch"] + train_data["SibSp"] + 1
test_data["FamilySize"] = test_data["Parch"] + test_data["SibSp"] + 1

train_data["IsAlone"] = 0
test_data["IsAlone"] = 0

train_data.loc[train_data["FamilySize"] == 1, "IsAlone"] = 1
test_data.loc[test_data["FamilySize"] == 1, "IsAlone"] = 1

In [18]:
train_median_fare = train_data["Fare"].median()
train_data["Fare"] = train_data["Fare"].fillna(train_median_fare)
test_data["Fare"] = test_data["Fare"].fillna(train_median_fare)

# Still need to learn about qcut()
train_data['FareBin'], fare_bins = pd.qcut(train_data['Fare'], 4, retbins=True, labels=['Cheap', 'Mid', 'High', 'Expensive'])
test_data['FareBin'] = pd.cut(test_data['Fare'], bins=fare_bins, labels=['Cheap', 'Mid', 'High', 'Expensive'], include_lowest=True)

In [19]:
y = train_data["Survived"]

features = ["Pclass", "AgeGroup", "FamilySize", "IsAlone", "FareBin", "Title"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

model = RandomForestClassifier(n_estimators = 100, random_state = 1, max_depth = 5)
model.fit(X, y)
predictions = model.predict(X_test)

out = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output = out.to_csv('submission.csv', index = False)
print("Your submission was successfully saved!")

NameError: name 'RandomForestClassifier' is not defined